# Panel c — dose–response relationship between ΔAge and outcomes of 12 systems

This notebook builds a continuous `ΔAge` from the OOF age predictions of the integrated four-omics clock and analyses the association of each 1 SD increase with all available system phenotypes. Standardized β and partial r are reported for continuous outcomes and OR for binary outcomes. All models adjust for chronological age and sex only and are **not adjusted for BMI**.

The primary test family is all successfully fitted phenotypes, adjusted together by Benjamini–Hochberg FDR; within-system FDR is also reported. The forest plot is grouped by system; point area shows the complete-case sample size, colour shows the system and filled points have global FDR < 0.05.

## 1. Import the analysis engine and set paths

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

HERE_CANDIDATES = (Path.cwd(), Path.cwd() / '02-papers' / 'Result-05')
HERE = next((p.resolve() for p in HERE_CANDIDATES if (p / 'dose_response_forest.py').exists()), None)
if HERE is None:
    raise FileNotFoundError('Cannot locate dose_response_forest.py; run from the project root or the notebook directory')

import sys
import importlib
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import dose_response_forest as dr
dr = importlib.reload(dr)  # reload the latest analysis code when this cell is rerun

CLOCK_PATH = dr.DEFAULT_CLOCK
SYSTEMS_DIR = "../01-interim_data/systems_data"          # or Path('.../systems_data')
SEX_COVARIATES = None       # optional CSV/parquet: participant_id, sex
MIN_N = 50
MIN_EVENTS = 20
TOP_N_PER_SYSTEM = 10
OUTPUT_DIR = HERE / 'outputs'
FIGURE_DIR = HERE / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

{'here': HERE, 'clock': CLOCK_PATH, 'systems_dir': SYSTEMS_DIR}

## 2. Build ΔAge of the integrated clock

OOF predictions are first aggregated across repeats per participant; `predicted age − chronological age` is then computed, residualized on chronological age and standardized to SD units within the complete clock cohort.

In [ ]:
clocks = dr.load_integrated_delta_age(CLOCK_PATH)
display(clocks.describe(include='all'))
print(f'Clock participants: {len(clocks):,}')
print(f"DeltaAge SD: {clocks['delta_age_sd'].std(ddof=1):.6f}")

## 3. Automatic audit of all phenotypes of the 12 systems

Apart from metadata columns (`dr.META_COLS`: participant ID, age, sex, BMI, dates, batch and so on), **every column of a system table is treated as a computable phenotype**: columns with exactly two values are modelled as binary, and all others are converted to numeric and modelled as continuous (non-numeric content becomes missing). Cases that cannot be fitted, such as insufficient sample size or zero variance, are written to the audit table at the modelling stage.

Instrument parameters such as DXA scan dose were removed from systems_data upstream by `Result-01/00-filter_system_features.py`.

In [ ]:
system_dir = dr.find_system_dir(SYSTEMS_DIR)
system_tables, inventory_audit = dr.load_system_tables(system_dir)

inventory_summary = (inventory_audit.groupby(['system', 'status'], dropna=False)
                     .size().unstack(fill_value=0).reindex(dr.SYSTEM_FILES, fill_value=0))
display(inventory_summary)
display(inventory_audit.head(20))

# Check against the retained-feature list of Result-01/00-filter_system_features.py; a warning is raised on mismatch
feature_check = dr.compare_with_feature_lists(inventory_audit)
feature_check.to_csv(OUTPUT_DIR / 'panel_c_feature_list_check.csv', index=False)
display(feature_check)

## 4. Fit age- and sex-adjusted models

Continuous outcomes: `z(outcome) ~ ΔAge_SD + age + sex`.  
Binary outcomes: `logit(P(outcome=1)) ~ ΔAge_SD + age + sex`.  
Repeated measurements are first aggregated per participant, using the median for continuous variables and the mode for binary variables. BMI enters no model.

In [ ]:
sex_covariates = dr.load_external_covariates(SEX_COVARIATES)
results, model_audit = dr.analyse(
    clocks, system_tables, sex_covariates, min_n=MIN_N, min_events=MIN_EVENTS
)
if results.empty:
    raise RuntimeError('No phenotype was fitted successfully; check the audit table below')

display(results[['system_label', 'label', 'outcome_type', 'n', 'effect',
                 'ci_low', 'ci_high', 'p_value', 'q_value_global',
                 'fdr_significant']])
print(f"Models: {len(results):,}; systems: {results['system'].nunique()}; "
      f"global FDR discoveries: {results['fdr_significant'].sum()}")

## 5. Save statistics and audit records

In [ ]:
audit = pd.concat([
    inventory_audit.assign(stage='inventory'),
    model_audit.assign(stage='model'),
], ignore_index=True, sort=False)

clocks.to_parquet(OUTPUT_DIR / 'panel_c_integrated_delta_age.parquet', index=False)
results.to_csv(OUTPUT_DIR / 'panel_c_dose_response_effects.csv', index=False)
audit.to_csv(OUTPUT_DIR / 'panel_c_dose_response_audit.csv', index=False)

diagnostics = {
    'clock_path': str(CLOCK_PATH),
    'systems_dir': str(system_dir),
    'phenotype_inventory': 'all non-metadata columns',
    'sex_covariates': str(SEX_COVARIATES) if SEX_COVARIATES else 'system tables',
    'n_clock_participants': len(clocks),
    'n_models': len(results),
    'n_systems': int(results['system'].nunique()),
    'n_continuous': int(results['outcome_type'].eq('continuous').sum()),
    'n_binary': int(results['outcome_type'].eq('binary').sum()),
    'n_global_fdr': int(results['fdr_significant'].sum()),
    'fdr_family': 'all fitted phenotypes',
    'covariates': ['chronological age', 'sex'],
    'bmi_adjusted': False,
    'display_rule': f'{TOP_N_PER_SYSTEM} smallest two-sided P values per system; all if fewer',
}
(OUTPUT_DIR / 'panel_c_dose_response_diagnostics.json').write_text(
    json.dumps(diagnostics, indent=2), encoding='utf-8'
)
display(audit.groupby(['stage', 'status']).size().rename('n').to_frame())

## 6. Draw the Panel c forest plot

In [ ]:
plot_results = dr.select_top_phenotypes(results, TOP_N_PER_SYSTEM)
plot_results.to_csv(OUTPUT_DIR / 'panel_c_dose_response_displayed_top10.csv', index=False)
dr.write_caption(
    plot_results, OUTPUT_DIR / 'panel_c_dose_response_caption.txt', total_tested=len(results)
)
display(plot_results.groupby('system_label').size().rename('displayed_phenotypes').to_frame())

figure_base = FIGURE_DIR / 'panel_c_dose_response_forest'
png_pages = dr.plot_forest(
    plot_results, figure_base, max_rows_per_page=max(10, len(plot_results)),
    selection_note='Up to 10 smallest-P phenotypes per system are displayed',
)
display(Image(filename=str(png_pages[0])))
print(f'PNG pages: {len(png_pages)}')
for path in png_pages:
    print(path)
print('Multipage PDF:', figure_base.with_suffix('.pdf'))